In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from share.sumcheck import Verifier, Prover
from share.mlesumcheck import MLEProver, trans_tables
from share.interactive import Simulation, Party
from share.field import F
from share.poly import UniPoly
from share.mle import MLE
from share.matmul import MatmulProver, MatmulVerifier

from itertools import product
from random import randint, seed
from types import MethodType
from math import prod

In [2]:
from random import random, seed

def random_graph(n, p=0.3):
    A = [0] * (n * n)
    for i in range(n):
        for j in range(i + 1, n):
            if random() < p:
                A[i * n + j] = A[j * n + i] = 1
    return A

seed(2026)
A_int = random_graph(64)
A = [F(e) for e in A_int]

class CountTrianglesSimpleProver(MLEProver):

    def __init__(self, name, A):
        ln = self.ln = (len(A).bit_length() - 1) // 2
        Axy = [0] * 2**(3*ln)
        Ayz = [0] * 2**(3*ln)
        Axz = [0] * 2**(3*ln)

        for i in range(2**(3*ln)):
            x = i >> 2*ln
            y = (i >> ln) ^ (x << ln)
            z = i ^ (x << 2*ln) ^ (y << ln)
            Axy[i] = A[(x << ln) | y]
            Ayz[i] = A[(y << ln) | z]
            Axz[i] = A[(x << ln) | z]

        super().__init__(name, [Axy, Ayz, Axz])


class CountTrianglesSimpleVerifier(Verifier):
    def __init__(self, name, A, claim):
        ln = self.ln = (len(A).bit_length() - 1) // 2
        mle_A = MLE(A)
        g = lambda *x: mle_A(*x[:-ln])*mle_A(*x[ln:])*mle_A(*x[:ln],*x[-ln:])
        degs = [3] * (3*ln) # loose for check (true is 2)
        super().__init__(name, g, degs, claim)

prover = CountTrianglesSimpleProver("P", A)
verifier = CountTrianglesSimpleVerifier("V", A, prover.sum())

prover.set_dst(verifier)

seed(1337)
sim = Simulation([prover, verifier])
sim.run()
print(sim.result)
sim.print_log()
sim.print_times()
print(prover.sum()/6)

{'P': None, 'V': True}
P -> V: ('g1', [F(2670), F(3384), F(3094), F(1800)])
V -> P: ('r1', F(2134139354768280835))
P -> V: ('g2', [F(819727441047378221), F(1419514358233824104), F(627820387427104229), F(750488537840912547)])
V -> P: ('r2', F(1636871794675947429))
P -> V: ('g3', [F(1785868614622235298), F(303437483439585850), F(222330274546259636), F(1542546987942256656)])
V -> P: ('r3', F(1316587892191542527))
P -> V: ('g4', [F(1371552962036474635), F(372841957116456886), F(291991976427279571), F(1129003019968942690)])
V -> P: ('r4', F(1687634588898568434))
P -> V: ('g5', [F(1906170054627100784), F(1485175293802365829), F(1587471688079877751), F(2213059237459636550)])
V -> P: ('r5', F(1787488633946274655))
P -> V: ('g6', [F(1995142895585004773), F(185931680121518005), F(151426331439168269), F(1891626849537955565)])
V -> P: ('r6', F(767359944638474350))
P -> V: ('g7', [F(1963166098315749981), F(731661032544310292), F(1740727105055537022), F(378678297422042269)])
V -> P: ('r7', F(1463919

In [3]:
class CountTrianglesEfficientProver(Party):

    def __init__(self, name, A):
        super().__init__(name)

        self.matmul = MatmulProver(name, A, A)
        self.mlesum = MLEProver(name, [self.matmul.C, A])

    def sum(self):
        return self.mlesum.sum()

    def set_dst(self, verifier):
        self.matmul.set_dst(verifier)
        self.mlesum.set_dst(verifier)

    def run(self):
        yield from self.mlesum.run()
        yield from self.matmul.run()


class CountTrianglesEfficientVerifier(Party):

    def __init__(self, name, A, claim):
        super().__init__(name)
        ln = self.ln = (len(A).bit_length() - 1) // 2
        self.A = A

        self.matmul = MatmulVerifier(name, (), A, A)
        self.mlesum = Verifier(name, lambda _: _, [2, 2] * ln, claim)

    def set_dst(self, prover):
        self.matmul.set_dst(prover)

    def run(self):
        out = yield from self.mlesum.loop()
        if out is None:
            return False
        rs, last = out
        
        ln = self.ln
        self.matmul.r = (rs[:ln], rs[ln:])
        result = yield from self.matmul.run()
        return (self.matmul.sum * MLE(self.A)(*rs) == last) and result


prover = CountTrianglesEfficientProver("P", A)
verifier = CountTrianglesEfficientVerifier("V", A, prover.sum())

prover.set_dst(verifier)
verifier.set_dst(prover)

seed(1337)
sim = Simulation([prover, verifier])
sim.run()
print(sim.result)
sim.print_log()
sim.print_times()
print(prover.sum()/6)

{'P': None, 'V': True}
P -> V: ('g1', [F(2670), F(3384), F(3094)])
V -> P: ('r1', F(2134139354768280835))
P -> V: ('g2', [F(819727441047378221), F(1419514358233824104), F(627820387427104229)])
V -> P: ('r2', F(1636871794675947429))
P -> V: ('g3', [F(1785868614622235298), F(303437483439585850), F(222330274546259636)])
V -> P: ('r3', F(1316587892191542527))
P -> V: ('g4', [F(1371552962036474635), F(372841957116456886), F(291991976427279571)])
V -> P: ('r4', F(1687634588898568434))
P -> V: ('g5', [F(1906170054627100784), F(1485175293802365829), F(1587471688079877751)])
V -> P: ('r5', F(1787488633946274655))
P -> V: ('g6', [F(1995142895585004773), F(185931680121518005), F(151426331439168269)])
V -> P: ('r6', F(767359944638474350))
P -> V: ('g7', [F(1963166098315749981), F(731661032544310292), F(1740727105055537022)])
V -> P: ('r7', F(1463919589942000245))
P -> V: ('g8', [F(2261005844593421807), F(137834390809704145), F(51962752670515012)])
V -> P: ('r8', F(829746912854020349))
P -> V: ('g9

In [4]:
class CountTrianglesOnePointProver(Party):

    def __init__(self, name, A):
        super().__init__(name)

        self.A = A
        self.matmul = MatmulProver(name, A, A)
        self.mlesum = MLEProver(name, [self.matmul.C, A])

    def sum(self):
        return self.mlesum.sum()

    def set_dst(self, verifier):
        self.matmul.set_dst(verifier)
        self.mlesum.set_dst(verifier)

    def run(self):
        yield from self.mlesum.run()
        yield from self.matmul.run()

        dst = self.matmul.dst
        _, (r1, r2, r3) = yield

        r1r2 = r1 + r2
        r2r3 = r2 + r3
        r1r3 = r1 + r3

        lp = [UniPoly([r1r2[i], r2r3[i], r1r3[i]]) for i in range(len(r1r2))]
        l  = lambda x: [p(x) for p in lp]

        g, degs = trans_tables([self.A])
        ev = [g(*l(i)) for i in range(2*len(r1r2) + 1)]

        yield dst, ev


class CountTrianglesOnePointVerifier(Party):

    def __init__(self, name, A, claim):
        super().__init__(name)
        ln = self.ln = (len(A).bit_length() - 1) // 2
        self.A = A

        self.matmul = MatmulVerifier(name, (), A, A)
        self.mlesum = Verifier(name, lambda _: _, [2, 2] * ln, claim)

    def set_dst(self, prover):
        self.matmul.set_dst(prover)

    def run(self):
        out = yield from self.mlesum.loop()
        if out is None:
            return False
        rs, c1 = out
        
        ln = self.ln
        self.matmul.r = (r1:=rs[:ln], r3:=rs[ln:])
        out = yield from self.matmul.loop()
        if out is None:
            return False
        r2, c2 = out

        dst = self.matmul.dst
        yield dst, (r1, r2, r3)
        _, ev = yield

        if ev[0]*ev[1] != c2 or self.matmul.sum * ev[2] != c1:
            return False
        
        r1r2 = r1 + r2
        r2r3 = r2 + r3
        r1r3 = r1 + r3

        rt = F(randint(1, F.p))
        r  = [UniPoly([r1r2[i], r2r3[i], r1r3[i]])(rt) for i in range(2*ln)]

        g, degs = trans_tables([self.A])

        return UniPoly(ev)(rt) == g(*r)

prover = CountTrianglesOnePointProver("P", A)
verifier = CountTrianglesOnePointVerifier("V", A, prover.sum())

prover.set_dst(verifier)
verifier.set_dst(prover)

seed(1337)
sim = Simulation([prover, verifier])
sim.run()
print(sim.result)
sim.print_log()
sim.print_times()
print(prover.sum()/6)

{'P': None, 'V': True}
P -> V: ('g1', [F(2670), F(3384), F(3094)])
V -> P: ('r1', F(2134139354768280835))
P -> V: ('g2', [F(819727441047378221), F(1419514358233824104), F(627820387427104229)])
V -> P: ('r2', F(1636871794675947429))
P -> V: ('g3', [F(1785868614622235298), F(303437483439585850), F(222330274546259636)])
V -> P: ('r3', F(1316587892191542527))
P -> V: ('g4', [F(1371552962036474635), F(372841957116456886), F(291991976427279571)])
V -> P: ('r4', F(1687634588898568434))
P -> V: ('g5', [F(1906170054627100784), F(1485175293802365829), F(1587471688079877751)])
V -> P: ('r5', F(1787488633946274655))
P -> V: ('g6', [F(1995142895585004773), F(185931680121518005), F(151426331439168269)])
V -> P: ('r6', F(767359944638474350))
P -> V: ('g7', [F(1963166098315749981), F(731661032544310292), F(1740727105055537022)])
V -> P: ('r7', F(1463919589942000245))
P -> V: ('g8', [F(2261005844593421807), F(137834390809704145), F(51962752670515012)])
V -> P: ('r8', F(829746912854020349))
P -> V: ('g9